In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

from config import load_config
from dataset_coco import COCOInstanceDataset
from models import Mask2Former
from lightning_module import SegmentationModule

In [ ]:
config = load_config('../configs/config_coco_mask2former_instance.json')

DATA_ROOT = config['data']['root_dir']
TRAIN_SPLIT = config['data']['split']
VAL_SPLIT = config['data']['val_split']
TEST_SPLIT = config['data']['test_split']
TARGET_SIZE = tuple(config['data']['target_size'])
BATCH_SIZE = config['data']['batch_size']
NUM_WORKERS = config['data']['num_workers']
MIN_AREA = config['data']['min_area']
MODE = config['data']['mode']
NUM_CLASSES = config['model']['num_classes']

MAX_EPOCHS = config['training']['max_epochs']
LEARNING_RATE = config['training']['learning_rate']

NUM_QUERIES = config['model']['num_queries']
HIDDEN_DIM = config['model']['hidden_dim']
NHEADS = config['model']['nheads']
NUM_DECODER_LAYERS = config['model']['num_decoder_layers']

print(f"Model: Mask2Former (Hybrid)")
print(f"Classes: {NUM_CLASSES}")


In [ ]:
def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    masks = [item[1] for item in batch]
    labels = [item[2] for item in batch]
    return images, masks, labels

train_dataset = COCOInstanceDataset(
    root_dir=DATA_ROOT,
    split=TRAIN_SPLIT,
    target_size=TARGET_SIZE,
    min_area=MIN_AREA,
    mode=MODE
)

val_dataset = COCOInstanceDataset(
    root_dir=DATA_ROOT,
    split=VAL_SPLIT,
    target_size=TARGET_SIZE,
    min_area=MIN_AREA,
    mode=MODE
)

test_dataset = COCOInstanceDataset(
    root_dir=DATA_ROOT,
    split=TEST_SPLIT,
    target_size=TARGET_SIZE,
    min_area=MIN_AREA,
    mode=MODE
)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples: {len(val_dataset)}')
print(f'Test samples: {len(test_dataset)}')


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_images, sample_masks, sample_labels = next(iter(train_loader))

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for i in range(min(2, len(sample_images))):
    img = sample_images[i].cpu().numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    
    axes[i*2].imshow(img)
    axes[i*2].set_title(f'Image {i+1}')
    axes[i*2].axis('off')
    
    combined_mask = np.zeros(sample_masks[i].shape[1:], dtype=np.int32)
    for j, mask in enumerate(sample_masks[i]):
        combined_mask[mask.numpy() > 0] = j + 1
    
    axes[i*2+1].imshow(combined_mask, cmap='tab20')
    axes[i*2+1].set_title(f'Instances: {len(sample_masks[i])}')
    axes[i*2+1].axis('off')

plt.tight_layout()
plt.show()

print(f"\nBatch info:")
print(f"Images: {sample_images.shape}")
print(f"Sample 0: {len(sample_masks[0])} instances, labels: {sample_labels[0].tolist()[:10]}...")


In [ ]:
model = Mask2Former(
    num_classes=NUM_CLASSES,
    num_queries=NUM_QUERIES,
    emb_dim=HIDDEN_DIM,
    nhead=NHEADS,
    nlayers=NUM_DECODER_LAYERS
)

model = SegmentationModule(
    model=model,
    num_classes=NUM_CLASSES,
    learning_rate=LEARNING_RATE,
    weight_decay=config['training']['weight_decay'],
    optimizer='adamw'
)

print(f"Mask2Former model created for instance segmentation")
print(f"  Transformer queries: {NUM_QUERIES}")
print(f"  Embedding dimension: {HIDDEN_DIM}")
print(f"  Attention heads: {NHEADS}")
print(f"  Decoder layers: {NUM_DECODER_LAYERS}")


In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=config['training']['checkpoint_dirpath'],
    filename=config['training']['checkpoint_filename'],
    save_top_k=config['training']['checkpoint_save_top_k'],
    monitor='val_loss',
    mode='min'
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=config['training']['early_stopping_patience'],
    mode='min'
)

logger = TensorBoardLogger(
    save_dir=config['training']['log_dir'],
    name=config['training']['experiment_name']
)


In [ ]:
trainer = pl.Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback, early_stopping],
    logger=logger,
    accelerator=config['hardware']['accelerator'],
    devices=config['hardware']['devices'],
    log_every_n_steps=config['training']['log_every_n_steps']
)


In [ ]:
trainer.fit(model, train_loader, val_loader)


# Testing: Semantic Segmentation (mIoU)

In [ ]:
from tqdm.notebook import tqdm
from torchmetrics import JaccardIndex

model.eval()
test_iou = JaccardIndex(
    task='multiclass',
    num_classes=NUM_CLASSES + 1,
    average='macro',
    ignore_index=0
).to(model.device)

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Semantic mIoU'):
        images, instance_masks, instance_labels = batch
        images = images.to(model.device)
        
        outputs = model(images)
        preds = model.model.postprocess(outputs, mode='semantic')
        
        semantic_targets = model._instance_to_semantic(
            instance_masks, instance_labels, device=model.device
        )
        
        test_iou.update(preds, semantic_targets)

semantic_miou = test_iou.compute()
print(f'\nSemantic Segmentation mIoU: {semantic_miou:.4f}')


# Testing: Instance Segmentation (mAP)

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

test_map = MeanAveragePrecision(iou_type='segm').to(model.device)

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Instance mAP'):
        images, instance_masks, instance_labels = batch
        images = images.to(model.device)
        
        outputs = model(images)
        preds = model.model.postprocess(outputs, mode='instance')
        
        preds_coco, targets_coco = [], []
        for b in range(len(images)):
            if len(preds[b]['classes']) > 0:
                pred_dict = {
                    'masks': (preds[b]['masks'] > 0.5).cpu().to(torch.uint8),
                    'labels': preds[b]['classes'].cpu(),
                    'scores': preds[b]['scores'].cpu()
                }
            else:
                pred_dict = {
                    'masks': torch.zeros((0, 360, 480), dtype=torch.uint8),
                    'labels': torch.tensor([], dtype=torch.long),
                    'scores': torch.tensor([], dtype=torch.float32)
                }
            preds_coco.append(pred_dict)
            
            valid_idx = instance_labels[b] > 0
            if valid_idx.any():
                gt_dict = {
                    'masks': (instance_masks[b][valid_idx] > 0.5).cpu().to(torch.uint8),
                    'labels': instance_labels[b][valid_idx].cpu()
                }
            else:
                gt_dict = {
                    'masks': torch.zeros((0, 360, 480), dtype=torch.uint8),
                    'labels': torch.tensor([], dtype=torch.long)
                }
            targets_coco.append(gt_dict)
        
        test_map.update(preds_coco, targets_coco)

map_metrics = test_map.compute()
print(f"\nInstance Segmentation Metrics:")
print(f"  mAP (IoU=0.50:0.95): {map_metrics['map']:.4f}")
print(f"  mAP@50:              {map_metrics['map_50']:.4f}")
print(f"  mAP@75:              {map_metrics['map_75']:.4f}")


# Visualization of Predictions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images_vis, instance_masks_vis, instance_labels_vis = next(iter(test_loader))
images_vis = images_vis[:4].to(model.device)

with torch.no_grad():
    outputs = model(images_vis)
    preds = model.model.postprocess(outputs, mode='instance')

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for i in range(4):
    img = images_vis[i].cpu().numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Original Image')
    axes[i, 0].axis('off')
    
    gt_vis = img.copy()
    for j in range(instance_masks_vis[i].shape[0]):
        if instance_labels_vis[i][j] > 0:
            mask = instance_masks_vis[i][j].numpy() > 0.5
            color = plt.cm.tab20(j % 20)[:3]
            for c in range(3):
                gt_vis[:, :, c][mask] = gt_vis[:, :, c][mask] * 0.5 + color[c] * 0.5
    axes[i, 1].imshow(gt_vis)
    axes[i, 1].set_title(f'GT ({instance_masks_vis[i].shape[0]} instances)')
    axes[i, 1].axis('off')
    
    pred_vis = img.copy()
    for j in range(len(preds[i]['masks'])):
        mask = preds[i]['masks'][j].cpu().numpy() > 0.5
        color = plt.cm.tab20(j % 20)[:3]
        for c in range(3):
            pred_vis[:, :, c][mask] = pred_vis[:, :, c][mask] * 0.5 + color[c] * 0.5
    axes[i, 2].imshow(pred_vis)
    axes[i, 2].set_title(f'Predicted ({len(preds[i]["masks"])} instances)')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()


# Internal Components Visualization

## 1. Position Embedding: Spatial Patterns

In [ ]:
import torch.nn.functional as F

images, instance_masks, instance_labels = next(iter(test_loader))
images = images[:1].to(model.device)

with torch.no_grad():
    enc1 = model.model.encoder1(images)
    enc2 = model.model.encoder2(enc1)
    enc3 = model.model.encoder3(enc2)
    enc4 = model.model.encoder4(enc3)
    enc5 = model.model.encoder5(enc4)
    bridge = model.model.bridge(enc5)
    dec1 = model.model.up1(bridge)
    dec1 = model.model._match_size(dec1, enc4)
    dec1 = torch.cat([dec1, enc4], dim=1)
    dec1 = model.model.dec1(dec1)
    dec2 = model.model.up2(dec1)
    dec2 = model.model._match_size(dec2, enc3)
    dec2 = torch.cat([dec2, enc3], dim=1)
    dec2 = model.model.dec2(dec2)
    features = model.model.feat_proj(dec2)
    pos_embed = model.model.pos_emb(features)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(8):
    ax = axes[i // 4, i % 4]
    ax.imshow(pos_embed[0, i * 32].cpu(), cmap='RdBu')
    ax.set_title(f'Channel {i*32}')
    ax.axis('off')
plt.suptitle('Position Embedding Channels (sin/cos patterns)', fontsize=14)
plt.tight_layout()
plt.show()
print(f'Position embedding shape: {pos_embed.shape}')


## 2. Attention Weights: What Query Sees

In [ ]:
with torch.no_grad():
    outputs = model(images)
    
    mem = features.flatten(2).permute(0, 2, 1)
    mem_pos = pos_embed.flatten(2).permute(0, 2, 1)
    queries = model.model.queries.weight.unsqueeze(0).expand(1, -1, -1)
    q_pos = model.model.query_pos.weight.unsqueeze(0).expand(1, -1, -1)
    
    Q = queries + q_pos
    K = mem + mem_pos
    attention = torch.matmul(Q, K.transpose(-2, -1)) / (Q.shape[-1] ** 0.5)
    attention = torch.softmax(attention, dim=-1)

H, W = features.shape[2:]
attention_maps = attention[0].view(100, H, W)

class_probs = torch.softmax(outputs['class_logits'][0], dim=-1)
person_probs = class_probs[:, 0]
top_queries = torch.argsort(person_probs, descending=True)[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, q in enumerate(top_queries):
    ax = axes[idx // 3, idx % 3]
    attn_map = attention_maps[q].cpu()
    attn_map = F.interpolate(attn_map.unsqueeze(0).unsqueeze(0), 
                             size=(360, 480), mode='bilinear')[0, 0]
    ax.imshow(attn_map, cmap='hot')
    ax.set_title(f'Query {q.item()}, confidence={person_probs[q]:.3f}')
    ax.axis('off')
plt.suptitle('Query-to-Pixel Attention Maps (top 6 confident queries)', fontsize=14)
plt.tight_layout()
plt.show()


## 3. Mask Predictions: Raw Mask Logits

In [ ]:
mask_logits = outputs['masks'][0]
mask_probs = torch.sigmoid(mask_logits)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, q in enumerate(top_queries):
    ax = axes[idx // 3, idx % 3]
    mask = mask_probs[q].cpu()
    ax.imshow(mask, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Query {q.item()}, confidence={person_probs[q]:.3f}')
    ax.axis('off')
plt.suptitle('Raw Mask Predictions (before postprocessing)', fontsize=14)
plt.tight_layout()
plt.show()
print(f'Mask logits shape: {mask_logits.shape}')
print(f'Mask probs range: [{mask_probs.min():.3f}, {mask_probs.max():.3f}]')


## 4. Hungarian Matching: Cost Matrix Heatmap

In [ ]:
from scipy.optimize import linear_sum_assignment

targets = model._prepare_targets([instance_masks[0]], [instance_labels[0]])

with torch.no_grad():
    class_probs = torch.softmax(outputs['class_logits'][0], dim=-1)
    mask_probs = torch.sigmoid(outputs['masks'][0])
    
    gt_labels = targets[0]['labels'].to(model.device)
    gt_masks = targets[0]['masks'].to(model.device)
    
    if len(gt_labels) > 0:
        cost_class = -class_probs[:, gt_labels]
        
        pred_flat = mask_probs.flatten(1)
        gt_flat = gt_masks.flatten(1)
        intersection = torch.matmul(pred_flat, gt_flat.t())
        pred_sum = pred_flat.sum(1, keepdim=True)
        gt_sum = gt_flat.sum(1, keepdim=True).t()
        cost_dice = 1 - (2 * intersection) / (pred_sum + gt_sum + 1e-8)
        
        cost_matrix = cost_class + cost_dice
        
        pred_idx, gt_idx = linear_sum_assignment(cost_matrix.cpu().numpy())
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        im1 = axes[0].imshow(cost_class.cpu(), aspect='auto', cmap='RdYlGn_r')
        axes[0].set_title('Cost: Class CE')
        axes[0].set_xlabel('GT instances')
        axes[0].set_ylabel('Queries')
        plt.colorbar(im1, ax=axes[0])
        
        im2 = axes[1].imshow(cost_dice.cpu(), aspect='auto', cmap='RdYlGn_r')
        axes[1].set_title('Cost: Dice Loss')
        axes[1].set_xlabel('GT instances')
        axes[1].set_ylabel('Queries')
        plt.colorbar(im2, ax=axes[1])
        
        im3 = axes[2].imshow(cost_matrix.cpu(), aspect='auto', cmap='RdYlGn_r')
        axes[2].set_title('Total Cost Matrix + Matches')
        axes[2].set_xlabel('GT instances')
        axes[2].set_ylabel('Queries')
        for p, g in zip(pred_idx, gt_idx):
            axes[2].plot(g, p, 'ro', markersize=10)
        plt.colorbar(im3, ax=axes[2])
        
        plt.suptitle('Hungarian Matching: Cost Matrix Visualization', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        print(f'Cost matrix shape: {cost_matrix.shape}')
        print(f'Num matches: {len(pred_idx)}')
        print(f'Matched queries: {pred_idx.tolist()}')
        print(f'Matched GT indices: {gt_idx.tolist()}')
    else:
        print('No GT instances in this image')
